In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lpad, concat_ws, sum as spark_sum, max as spark_max, expr, regexp_replace

spark = SparkSession.builder.getOrCreate()

# Leer el fichero
df = spark.read.option("header", True) \
               .option("sep", ";") \
               .option("inferSchema", False) \
               .csv("work/calidad_aire_datos_meteo_mes.csv")

# Filtrar magnitud 89 (precipitación)
df_prec = df.filter(col("MAGNITUD") == "89")

# Crear fecha YYYY-MM-DD
df_prec = df_prec.withColumn(
    "FECHA",
    concat_ws(
        "-",
        col("ANO"),
        lpad(col("MES"), 2, "0"),
        lpad(col("DIA"), 2, "0")
    )
)

# Convertir las 24 horas en filas
df_horas = df_prec.select(
    "FECHA",
    "MUNICIPIO",
    "ESTACION",
    expr("""
        stack(24,
            '01', H01, V01,
            '02', H02, V02,
            '03', H03, V03,
            '04', H04, V04,
            '05', H05, V05,
            '06', H06, V06,
            '07', H07, V07,
            '08', H08, V08,
            '09', H09, V09,
            '10', H10, V10,
            '11', H11, V11,
            '12', H12, V12,
            '13', H13, V13,
            '14', H14, V14,
            '15', H15, V15,
            '16', H16, V16,
            '17', H17, V17,
            '18', H18, V18,
            '19', H19, V19,
            '20', H20, V20,
            '21', H21, V21,
            '22', H22, V22,
            '23', H23, V23,
            '24', H24, V24
        ) as (HORA, PRECIP, VALIDACION)
    """)
)

# Filtrar validación V y convertir precipitación a double
df_validas = df_horas.filter(col("VALIDACION") == "V") \
    .withColumn("PRECIP", regexp_replace(col("PRECIP"), ",", ".")) \
    .withColumn("PRECIP", expr("try_cast(PRECIP as double)")) \
    .filter(col("PRECIP").isNotNull())

# Sumar precipitación diaria por estación
df_diario = df_validas.groupBy("FECHA", "MUNICIPIO", "ESTACION") \
    .agg(spark_sum("PRECIP").alias("PRECIPITACION_TOTAL"))

# Máximo por día
df_max_dia = df_diario.groupBy("FECHA") \
    .agg(spark_max("PRECIPITACION_TOTAL").alias("MAX_PRECIP_DIA"))

# Obtener estación con mayor precipitación cada día
d = df_diario.alias("d")
m = df_max_dia.alias("m")

resultado = d.join(
    m,
    (col("d.FECHA") == col("m.FECHA")) &
    (col("d.PRECIPITACION_TOTAL") == col("m.MAX_PRECIP_DIA")),
    "inner"
).select(
    col("d.FECHA").alias("FECHA"),
    col("d.MUNICIPIO").alias("MUNICIPIO"),
    col("d.ESTACION").alias("ESTACION"),
    col("d.PRECIPITACION_TOTAL").alias("PRECIPITACION_TOTAL")
).orderBy("FECHA")

print("Mayor precipitacion total por cada dia:")
resultado.show(truncate=False)

# Mayor precipitación de todo el periodo
max_total = resultado.agg(
    spark_max("PRECIPITACION_TOTAL").alias("MAX_TOTAL")
).alias("mt")

r = resultado.alias("r")

resultado_max_total = r.join(
    max_total,
    col("r.PRECIPITACION_TOTAL") == col("mt.MAX_TOTAL"),
    "inner"
).select(
    col("r.FECHA"),
    col("r.MUNICIPIO"),
    col("r.ESTACION"),
    col("r.PRECIPITACION_TOTAL")
)

print("Mayor precipitacion diaria de todo el periodo:")
resultado_max_total.show(truncate=False)

Mayor precipitación total por cada día:
+----------+---------+--------+-------------------+
|FECHA     |MUNICIPIO|ESTACION|PRECIPITACION_TOTAL|
+----------+---------+--------+-------------------+
|2026-02-01|120      |1       |20.8               |
|2026-02-02|161      |1       |18.4               |
|2026-02-03|127      |4       |7.6                |
|2026-02-04|45       |2       |11.600000000000001 |
|2026-02-05|115      |3       |30.800000000000004 |
|2026-02-06|67       |1       |6.3                |
|2026-02-07|115      |3       |21.799999999999997 |
|2026-02-08|120      |1       |25.100000000000005 |
+----------+---------+--------+-------------------+

Mayor precipitación diaria de todo el periodo:
+----------+---------+--------+-------------------+
|FECHA     |MUNICIPIO|ESTACION|PRECIPITACION_TOTAL|
+----------+---------+--------+-------------------+
|2026-02-05|115      |3       |30.800000000000004 |
+----------+---------+--------+-------------------+

